In [6]:
"""
MODELO 7/12 — MLP (Multi-Layer Perceptron) — VERSÃO OTIMIZADA

Mesma lógica e parâmetros do _IM_MLP.R, mas otimizado:
- Constrói o modelo Keras UMA VEZ por ticker
- A cada rodada WFA, reseta os pesos (equivalente a modelo novo)
- Evita retracing do tf.function (principal gargalo)

Os resultados são equivalentes a criar modelo novo a cada rodada,
pois pesos reinicializados aleatoriamente = modelo novo.

Uso:
    python 04_model_MLP.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICSMI")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"

TRAIN_SIZE = 250
TEST_SIZE = 5

# Parâmetros do R (deepnet::nn.train)
HIDDEN_LAYERS = [25, 15, 10, 5]
LEARNING_RATE = 0.01
EPOCHS = 5
BATCH_SIZE = 200
VISIBLE_DROPOUT = 0.2
HIDDEN_DROPOUT = 0.2
# ========================================================

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, InputLayer
from tensorflow.keras.optimizers import Adam

tf.get_logger().setLevel("ERROR")
tf.random.set_seed(42)
np.random.seed(42)


def build_mlp(input_dim: int) -> Sequential:
    """Constrói MLP uma vez. Pesos serão resetados a cada rodada WFA."""
    model = Sequential()
    model.add(InputLayer(shape=(input_dim,)))

    if VISIBLE_DROPOUT > 0:
        model.add(Dropout(VISIBLE_DROPOUT))

    for h in HIDDEN_LAYERS:
        model.add(Dense(h, activation="sigmoid"))
        if HIDDEN_DROPOUT > 0:
            model.add(Dropout(HIDDEN_DROPOUT))

    model.add(Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
    )
    return model


def reset_weights(model):
    """
    Reinicializa todos os pesos do modelo.
    Equivalente a criar um modelo novo do zero.
    """
    for layer in model.layers:
        if hasattr(layer, "kernel_initializer") and hasattr(layer, "kernel"):
            layer.kernel.assign(layer.kernel_initializer(layer.kernel.shape))
        if hasattr(layer, "bias_initializer") and hasattr(layer, "bias"):
            layer.bias.assign(layer.bias_initializer(layer.bias.shape))
    # Resetar estado do otimizador (Adam mantém momentos das iterações anteriores)
    model.optimizer = Adam(learning_rate=LEARNING_RATE)


def read_codes(path: Path) -> list:
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
    return df["Code"].str.strip().str.upper().tolist()


def run_wfa_mlp(code: str, base_dir: Path) -> dict:
    """Executa Walk-Forward Analysis com MLP para um ticker."""
    infile = base_dir / f"{code}_DatasetNew_MI.csv"
    outfile = base_dir / f"{code}_TradeSignal_MLP_MI.csv"

    if outfile.exists():
        return {"Code": code, "status": "skipped", "signals": 0}

    if not infile.exists():
        return {"Code": code, "status": "no_DatasetNew", "signals": 0}

    try:
        df = pd.read_csv(infile, encoding="utf-8-sig")
    except Exception as e:
        return {"Code": code, "status": f"read_error: {e}", "signals": 0}

    if df.shape[1] < 3:
        return {"Code": code, "status": "too_few_columns", "signals": 0}

    date_col = df.columns[0]
    label_col = df.columns[-1]
    feature_cols = df.columns[1:-1].tolist()
    input_dim = len(feature_cols)

    # --- Alinhamento WFA ---
    M = len(df)
    if M < TRAIN_SIZE + TEST_SIZE:
        return {"Code": code, "status": f"too_few_rows ({M})", "signals": 0}

    Q = (M - TRAIN_SIZE) // TEST_SIZE
    H = (M - TRAIN_SIZE) - TEST_SIZE * Q
    df = df.iloc[H:].reset_index(drop=True)

    dates = df[date_col].values
    X = df[feature_cols].values.astype(float)
    y = df[label_col].values.astype(int)

    # Construir modelo UMA VEZ por ticker (otimização)
    model = build_mlp(input_dim)

    predict_signal = []
    predict_dates = []

    # --- Loop WFA ---
    for i in range(Q):
        train_start = i * TEST_SIZE
        train_end = train_start + TRAIN_SIZE
        test_start = train_end
        test_end = test_start + TEST_SIZE

        X_train = X[train_start:train_end]
        y_train = y[train_start:train_end]
        X_test = X[test_start:test_end]
        test_dates_i = dates[test_start:test_end]

        if len(np.unique(y_train)) < 2:
            preds = [int(y_train[0])] * len(X_test)
        else:
            try:
                # Resetar pesos (equivalente a modelo novo)
                reset_weights(model)

                model.fit(
                    X_train, y_train,
                    epochs=EPOCHS,
                    batch_size=min(BATCH_SIZE, len(X_train)),
                    verbose=0,
                )

                probs = model.predict(X_test, verbose=0).reshape(-1)
                preds = (probs >= 0.5).astype(int).tolist()
            except Exception:
                preds = [0] * len(X_test)

        predict_signal.extend(preds)
        predict_dates.extend(test_dates_i)

    # Limpar sessão após cada ticker
    tf.keras.backend.clear_session()

    # --- Salvar ---
    if predict_signal:
        df_out = pd.DataFrame({"Date": predict_dates, "pre_signal": predict_signal})
        df_out.to_csv(outfile, index=False, encoding="utf-8-sig")

    return {"Code": code, "status": "ok", "signals": len(predict_signal)}


def main():
    codes = read_codes(SEC_NAMES)
    print(f"Modelo: MLP (hidden={HIDDEN_LAYERS}, sigmoid, lr={LEARNING_RATE})")
    print(f"WFA: d1={TRAIN_SIZE}, d2={TEST_SIZE}")
    print(f"Epochs={EPOCHS}, batch={BATCH_SIZE}")
    print(f"Otimização: reset de pesos (sem recompilação de grafo)")
    print(f"Tickers: {len(codes)}\n")

    report = []
    for code in tqdm(codes, desc="MLP Walk-Forward"):
        result = run_wfa_mlp(code, BASE_DIR)
        report.append(result)

    report_df = pd.DataFrame(report)
    report_df.to_csv(BASE_DIR / "model_MLP_report.csv", index=False, encoding="utf-8-sig")

    n_ok = (report_df["status"] == "ok").sum()
    n_skip = (report_df["status"] == "skipped").sum()
    avg_signals = report_df.loc[report_df["status"] == "ok", "signals"].mean()

    print(f"\n{'='*50}")
    print(f"Concluído: {n_ok} processados, {n_skip} já existiam.")
    print(f"Média de sinais por ação: {avg_signals:.0f}")
    print(f"Relatório: model_MLP_report.csv")


if __name__ == "__main__":
    main()

Modelo: MLP (hidden=[25, 15, 10, 5], sigmoid, lr=0.01)
WFA: d1=250, d2=5
Epochs=5, batch=200
Otimização: reset de pesos (sem recompilação de grafo)
Tickers: 239



MLP Walk-Forward: 100%|██████████| 239/239 [2:53:34<00:00, 43.58s/it]  


Concluído: 239 processados, 0 já existiam.
Média de sinais por ação: 564
Relatório: model_MLP_report.csv
